In [4]:
## 1. Import Libraries and Set Reproducibility

import os
import random

import numpy as np
import pandas as pd
import tensorflow as tf

# Fixed random seed
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)

# Request deterministic TensorFlow operations
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print("Random seed set to:", RANDOM_SEED)

Random seed set to: 42


In [5]:
## 2. Mount Google Drive and Set External Validation Paths

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = "/content/drive/MyDrive/Master_datascience_thesis"

# Existing preprocessing outputs
preprocessing_dir = os.path.join(PROJECT_ROOT,"outputs","preprocessing")

# Full Shenzhen external dataset
shenzhen_dir = os.path.join(PROJECT_ROOT,"data","Shenzhen","CXR_png")

# External validation outputs
external_output_dir = os.path.join(PROJECT_ROOT,"outputs","external_validation")

os.makedirs(external_output_dir, exist_ok=True)

# Load final TBX11K manifests
train_df = pd.read_csv(os.path.join(preprocessing_dir, "final_train_metadata.csv"))

validation_df = pd.read_csv(os.path.join(preprocessing_dir, "final_validation_metadata.csv"))

print("Final training images:", len(train_df))
print("Final validation images:", len(validation_df))
print("Shenzhen folder found:", os.path.exists(shenzhen_dir))
print("External validation output folder ready:", os.path.exists(external_output_dir))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Final training images: 6480
Final validation images: 1794
Shenzhen folder found: True
External validation output folder ready: True


In [6]:
## 3. Build Shenzhen External Validation Metadata

# Get all Shenzhen PNG images
shenzhen_files = sorted([file for file in os.listdir(shenzhen_dir)if file.lower().endswith(".png")])

# Build metadata
shenzhen_df = pd.DataFrame({"filename": shenzhen_files})

# Full image path
shenzhen_df["path"] = shenzhen_df["filename"].apply(lambda x: os.path.join(shenzhen_dir, x))

# Label from official filename convention
# 0 = normal / Non-TB
# 1 = abnormal / TB
shenzhen_df["label"] = shenzhen_df["filename"].apply(lambda x: 1 if x.endswith("_1.png") else 0)

shenzhen_df["binary_label"] = shenzhen_df["label"].map({
    0: "Non-TB",
    1: "TB"
})

print("Total Shenzhen images:", len(shenzhen_df))
print("\nClass counts:")
print(shenzhen_df["binary_label"].value_counts())

display(shenzhen_df.head())

Total Shenzhen images: 662

Class counts:
binary_label
TB        336
Non-TB    326
Name: count, dtype: int64


,filename,path,label,binary_label
0,CHNCXR_0001_0.png,/content/drive/MyDrive/Master_datascience_thes...,0,Non-TB
1,CHNCXR_0002_0.png,/content/drive/MyDrive/Master_datascience_thes...,0,Non-TB
2,CHNCXR_0003_0.png,/content/drive/MyDrive/Master_datascience_thes...,0,Non-TB
3,CHNCXR_0004_0.png,/content/drive/MyDrive/Master_datascience_thes...,0,Non-TB
4,CHNCXR_0005_0.png,/content/drive/MyDrive/Master_datascience_thes...,0,Non-TB


In [7]:
## 4. Check TBX11K-Shenzhen Image Overlap

import hashlib
import zipfile
from io import BytesIO
from PIL import Image

zip_path = os.path.join(PROJECT_ROOT,"data","TBX11K.zip")

# Create a comparable image-content hash
def external_overlap_hash(image):
    image = image.convert("L").resize((512, 512))
    return hashlib.sha256(image.tobytes()).hexdigest()


# Hash final TBX11K training and validation images
tbx_hashes = set()

with zipfile.ZipFile(zip_path, "r") as zip_file:

    development_paths = pd.concat([train_df["path"],validation_df["path"]])

    for path in development_paths:

        full_path = "TBX11K/imgs/" + path

        image_bytes = zip_file.read(full_path)

        with Image.open(BytesIO(image_bytes)) as image:
            tbx_hashes.add(external_overlap_hash(image))


# Compare Shenzhen images
overlap_files = []

for path in shenzhen_df["path"]:

    with Image.open(path) as image:
        image_hash = external_overlap_hash(image)

    if image_hash in tbx_hashes:
        overlap_files.append(path)


print("TBX11K development images checked:", len(development_paths))
print("Shenzhen images checked:", len(shenzhen_df))
print("TBX11K-Shenzhen overlaps:", len(overlap_files))

TBX11K development images checked: 8274
Shenzhen images checked: 662
TBX11K-Shenzhen overlaps: 0


In [8]:
## 5. Check Shenzhen Image Integrity

image_widths = []
image_heights = []
image_modes = []
image_formats = []
corrupted_files = []

for path in shenzhen_df["path"]:

    try:
        with Image.open(path) as image:
            image.load()

            image_widths.append(image.width)
            image_heights.append(image.height)
            image_modes.append(image.mode)
            image_formats.append(image.format)

    except Exception:
        corrupted_files.append(path)

print("Images checked:", len(shenzhen_df))
print("Corrupted/unreadable images:", len(corrupted_files))

print("\nImage modes:")
print(pd.Series(image_modes).value_counts())

print("\nImage formats:")
print(pd.Series(image_formats).value_counts())

print("\nWidth range:",min(image_widths), "to", max(image_widths))

print("Height range:",min(image_heights), "to", max(image_heights))

Images checked: 662
Corrupted/unreadable images: 0

Image modes:
P      635
RGB     27
Name: count, dtype: int64

Image formats:
PNG    662
Name: count, dtype: int64

Width range: 1130 to 3001
Height range: 948 to 3001


In [9]:
## 6. Save Shenzhen External Validation Metadata

shenzhen_metadata = shenzhen_df[["filename", "path", "label", "binary_label"]].copy()

shenzhen_metadata_path = os.path.join(external_output_dir,"shenzhen_external_metadata.csv")

shenzhen_metadata.to_csv( shenzhen_metadata_path, index=False)

print("Metadata saved:", shenzhen_metadata_path)
print("Rows saved:", len(shenzhen_metadata))

Metadata saved: /content/drive/MyDrive/Master_datascience_thesis/outputs/external_validation/shenzhen_external_metadata.csv
Rows saved: 662


In [10]:
## 7. Load the Selected SVM Model

import joblib
from skimage.feature import hog

traditional_ml_dir = os.path.join(PROJECT_ROOT, "outputs", "traditional_ml")

svm_model_path = os.path.join( traditional_ml_dir, "best_svm_hog_model.pkl")

# Fixed preprocessing from Notebook 04
IMAGE_SIZE = (224, 224)

HOG_ORIENTATIONS = 9
HOG_PIXELS_PER_CELL = (16, 16)
HOG_CELLS_PER_BLOCK = (2, 2)

# Locked threshold from Notebook 06
SVM_THRESHOLD = -0.491067

# Load the already-trained SVM pipeline
svm_model = joblib.load(svm_model_path)

print("SVM model loaded:", os.path.exists(svm_model_path))
print("Image size:", IMAGE_SIZE)
print("SVM threshold:", SVM_THRESHOLD)

SVM model loaded: True
Image size: (224, 224)
SVM threshold: -0.491067


In [11]:
## 8. Extract HOG Features from Shenzhen Images

from tqdm.auto import tqdm

def load_shenzhen_svm_image(image_path):

    with Image.open(image_path) as image:
        image = image.convert("L")
        image = image.resize(IMAGE_SIZE,Image.Resampling.BILINEAR)

    return np.array(image)


shenzhen_hog_features = np.empty((len(shenzhen_metadata), 6084),dtype=np.float32)

for index, image_path in enumerate(tqdm(shenzhen_metadata["path"])):

    image = load_shenzhen_svm_image(image_path)

    shenzhen_hog_features[index] = hog(
        image,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_PER_CELL,
        cells_per_block=HOG_CELLS_PER_BLOCK,
        block_norm="L2-Hys",
        feature_vector=True
    )


print("Shenzhen images processed:", len(shenzhen_metadata))
print("HOG feature shape:", shenzhen_hog_features.shape)

  0%|          | 0/662 [00:00<?, ?it/s]

Shenzhen images processed: 662
HOG feature shape: (662, 6084)


In [12]:
## 9. Generate SVM Predictions on Shenzhen

# Continuous SVM decision scores
shenzhen_svm_scores = svm_model.decision_function(shenzhen_hog_features)

# Apply the fixed threshold selected in Notebook 06
shenzhen_svm_predictions = (shenzhen_svm_scores >= SVM_THRESHOLD).astype(int)

print("Predictions generated:", len(shenzhen_svm_predictions))
print("Predicted Non-TB:", np.sum(shenzhen_svm_predictions == 0))
print("Predicted TB:", np.sum(shenzhen_svm_predictions == 1))

Predictions generated: 662
Predicted Non-TB: 300
Predicted TB: 362


In [13]:
## 10. Calculate SVM External Validation Metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

shenzhen_true_labels = shenzhen_metadata["label"].to_numpy()

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(
    shenzhen_true_labels,
    shenzhen_svm_predictions
).ravel()

specificity = tn / (tn + fp)

svm_external_metrics = {
    "Accuracy": accuracy_score(
        shenzhen_true_labels,
        shenzhen_svm_predictions
    ),
    "Precision": precision_score(
        shenzhen_true_labels,
        shenzhen_svm_predictions
    ),
    "Sensitivity": recall_score(
        shenzhen_true_labels,
        shenzhen_svm_predictions
    ),
    "Specificity": specificity,
    "F1_score": f1_score(
        shenzhen_true_labels,
        shenzhen_svm_predictions
    ),
    "AUROC": roc_auc_score(
        shenzhen_true_labels,
        shenzhen_svm_scores
    ),
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp
}

for metric, value in svm_external_metrics.items():
    if metric in ["TN", "FP", "FN", "TP"]:
        print(metric, ":", value)
    else:
        print(metric, ":", round(value, 4))

Accuracy : 0.574
Precision : 0.5746
Sensitivity : 0.619
Specificity : 0.5276
F1_score : 0.596
AUROC : 0.5907
TN : 172
FP : 154
FN : 128
TP : 208


In [14]:
## 11. Save SVM External Validation Results

svm_external_results = pd.DataFrame([{ "Model": "SVM", "Threshold": SVM_THRESHOLD, **svm_external_metrics}])

svm_external_results_path = os.path.join( external_output_dir, "svm_shenzhen_results.csv")

svm_external_results.to_csv(svm_external_results_path,index=False)

print("SVM external results saved:", svm_external_results_path)

display(svm_external_results)

SVM external results saved: /content/drive/MyDrive/Master_datascience_thesis/outputs/external_validation/svm_shenzhen_results.csv


,Model,Threshold,Accuracy,Precision,Sensitivity,Specificity,F1_score,AUROC,TN,FP,FN,TP
0,SVM,-0.491067,0.574018,0.574586,0.619048,0.527607,0.595989,0.590701,172,154,128,208


In [15]:
## 12. Prepare Shenzhen Images for ResNet50

BATCH_SIZE = 32

def load_shenzhen_resnet_image(image_path):

    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, (224, 224))

    return image


shenzhen_resnet_data = tf.data.Dataset.from_tensor_slices(shenzhen_metadata["path"].values)

shenzhen_resnet_data = shenzhen_resnet_data.map(load_shenzhen_resnet_image,num_parallel_calls=tf.data.AUTOTUNE)

shenzhen_resnet_data = shenzhen_resnet_data.batch(BATCH_SIZE)

print("Shenzhen images ready:", len(shenzhen_metadata))

Shenzhen images ready: 662


In [16]:
## 13. Load Fine-Tuned ResNet50

deep_learning_dir = os.path.join(PROJECT_ROOT,"outputs","deep_learning")

# Identify the saved fine-tuned ResNet50
resnet_files = [
    file for file in os.listdir(deep_learning_dir)
    if file.lower().endswith(".keras")
    and "resnet" in file.lower()
    and ("fine" in file.lower() or "tuned" in file.lower())
]

if len(resnet_files) != 1:
    raise ValueError(f"Expected one fine-tuned ResNet50 model, found: {resnet_files}")

resnet50_model_path = os.path.join(deep_learning_dir,resnet_files[0])

resnet50_model = tf.keras.models.load_model(resnet50_model_path)

# Locked threshold from Notebook 06
RESNET50_THRESHOLD = 0.698488

print("Model loaded:", resnet_files[0])
print("Threshold:", RESNET50_THRESHOLD)

Model loaded: best_resnet50_finetuned.keras
Threshold: 0.698488


In [17]:
## 14. Generate Fine-Tuned ResNet50 Predictions on Shenzhen

# Generate continuous probability scores
shenzhen_resnet_scores = resnet50_model.predict(shenzhen_resnet_data,verbose=1).ravel()

# Apply the fixed threshold selected in Notebook 06
shenzhen_resnet_predictions = (shenzhen_resnet_scores >= RESNET50_THRESHOLD).astype(int)

print("Predictions generated:", len(shenzhen_resnet_predictions))
print("Predicted Non-TB:", np.sum(shenzhen_resnet_predictions == 0))
print("Predicted TB:", np.sum(shenzhen_resnet_predictions == 1))

21/21 ━━━━━━━━━━━━━━━━━━━━ 217s 9s/step
Predictions generated: 662
Predicted Non-TB: 556
Predicted TB: 106


In [18]:
## 15. Calculate Fine-Tuned ResNet50 External Validation Metrics

shenzhen_true_labels = shenzhen_metadata["label"].to_numpy()

tn, fp, fn, tp = confusion_matrix(
    shenzhen_true_labels,
    shenzhen_resnet_predictions
).ravel()

specificity = tn / (tn + fp)

resnet_external_metrics = {
    "Accuracy": accuracy_score(
        shenzhen_true_labels,
        shenzhen_resnet_predictions
    ),
    "Precision": precision_score(
        shenzhen_true_labels,
        shenzhen_resnet_predictions
    ),
    "Sensitivity": recall_score(
        shenzhen_true_labels,
        shenzhen_resnet_predictions
    ),
    "Specificity": specificity,
    "F1_score": f1_score(
        shenzhen_true_labels,
        shenzhen_resnet_predictions
    ),
    "AUROC": roc_auc_score(
        shenzhen_true_labels,
        shenzhen_resnet_scores
    ),
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp
}

for metric, value in resnet_external_metrics.items():

    if metric in ["TN", "FP", "FN", "TP"]:
        print(metric, ":", value)
    else:
        print(metric, ":", round(value, 4))

Accuracy : 0.577
Precision : 0.7642
Sensitivity : 0.2411
Specificity : 0.9233
F1_score : 0.3665
AUROC : 0.64
TN : 301
FP : 25
FN : 255
TP : 81


In [19]:
## 16. Save Fine-Tuned ResNet50 External Validation Results

resnet_external_results = pd.DataFrame([{
    "Model": "Fine-tuned ResNet50",
    "Threshold": RESNET50_THRESHOLD,
    **resnet_external_metrics
}])

resnet_external_results_path = os.path.join(
    external_output_dir,
    "resnet50_finetuned_shenzhen_results.csv"
)

resnet_external_results.to_csv(
    resnet_external_results_path,
    index=False
)

print(
    "ResNet50 external results saved:",
    resnet_external_results_path
)

display(resnet_external_results)

ResNet50 external results saved: /content/drive/MyDrive/Master_datascience_thesis/outputs/external_validation/resnet50_finetuned_shenzhen_results.csv


,Model,Threshold,Accuracy,Precision,Sensitivity,Specificity,F1_score,AUROC,TN,FP,FN,TP
0,Fine-tuned ResNet50,0.698488,0.577039,0.764151,0.241071,0.923313,0.366516,0.640027,301,25,255,81


In [20]:
## 17. Compare External Validation Results

external_comparison = pd.concat(
    [
        svm_external_results,
        resnet_external_results
    ],
    ignore_index=True
)

comparison_path = os.path.join(
    external_output_dir,
    "shenzhen_model_comparison.csv"
)

external_comparison.to_csv(
    comparison_path,
    index=False
)

display(external_comparison)

print("Comparison saved:", comparison_path)

,Model,Threshold,Accuracy,Precision,Sensitivity,Specificity,F1_score,AUROC,TN,FP,FN,TP
0,SVM,-0.491067,0.574018,0.574586,0.619048,0.527607,0.595989,0.590701,172,154,128,208
1,Fine-tuned ResNet50,0.698488,0.577039,0.764151,0.241071,0.923313,0.366516,0.640027,301,25,255,81


Comparison saved: /content/drive/MyDrive/Master_datascience_thesis/outputs/external_validation/shenzhen_model_comparison.csv


In [21]:
## 18. Save Per-Image Shenzhen Predictions

shenzhen_predictions = shenzhen_metadata.copy()

shenzhen_predictions["svm_score"] = shenzhen_svm_scores
shenzhen_predictions["svm_prediction"] = shenzhen_svm_predictions

shenzhen_predictions["resnet50_score"] = shenzhen_resnet_scores
shenzhen_predictions["resnet50_prediction"] = shenzhen_resnet_predictions

predictions_path = os.path.join(external_output_dir,"shenzhen_predictions.csv")

shenzhen_predictions.to_csv(predictions_path,index=False)

print("Predictions saved:", predictions_path)
print("Rows saved:", len(shenzhen_predictions))

Predictions saved: /content/drive/MyDrive/Master_datascience_thesis/outputs/external_validation/shenzhen_predictions.csv
Rows saved: 662
